# 04 — Multi-Category Liver Pathology ModelsFor every liver pathology category with sufficient samples, run blood-based CV.

In [ ]:
import numpy as npimport pandas as pdfrom collections import Counterfrom joblib import Parallel, delayedfrom gtex_biomarkers.config import Configfrom gtex_biomarkers.data import (    load_raw_data, filter_whole_blood,    build_blood_expression_matrix, build_blood_subjid,)from gtex_biomarkers.models import run_cv, make_lr_pipelinefrom gtex_biomarkers.evaluation import plot_roc_grid, plot_pr_grid, plot_cm_grid, plot_boxplot_gridConfig.ensure_dirs()df_expr, df_samples, df_age, df_meta_url = load_raw_data()blood_meta = filter_whole_blood(df_samples)X_wb, _ = build_blood_expression_matrix(df_expr, blood_meta)blood_subjid = build_blood_subjid(X_wb)

## Discover Eligible Categories

In [ ]:
liver_imp = pd.read_csv(Config.PROCESSED_DIR / "liver_pathology_labels_imputed.csv")cat_counts = Counter()for val in liver_imp["Pathology.Categories.Final"].dropna():    for c in val.split(", "):        cat_counts[c.strip()] += 1eligible_cats = sorted(c for c, n in cat_counts.items()                       if n >= Config.SAMPLE_THRESHOLD_LIVER                       and c not in Config.NORMAL_LABELS)print(f"Eligible categories: {eligible_cats}")

## Build Donor Labels & Run Parallel CV

In [ ]:
# Donor-level labelsliver_known = liver_imp[liver_imp["Pathology.Categories.Final"].notna()].copy()all_donors = liver_imp["SUBJID"].unique()donor_labels = {}for cat in eligible_cats:    has_cat = liver_known["Pathology.Categories.Final"].str.contains(cat, case=False).astype(int)    donor_labels[cat] = has_cat.groupby(liver_known["SUBJID"]).max()# Run each category (parallelized)def _run_cat(cat):    y_cat = blood_subjid.map(donor_labels[cat])    keep = y_cat.notna()    res = run_cv(X_wb.loc[keep], y_cat.loc[keep].astype(int),                 blood_subjid.loc[keep].astype(str), make_lr_pipeline)    return (cat, res)results_list = Parallel(n_jobs=-1, verbose=10)(    delayed(_run_cat)(cat) for cat in eligible_cats)model_results = dict(results_list)for cat, res in model_results.items():    print(f"  {cat:20s}  AUC = {res['mean_auc']:.3f} ± {res['std_auc']:.3f}")

## Evaluation Plots

In [ ]:
plot_roc_grid(model_results, suptitle="Liver Pathology — ROC (5-fold CV)",             save_path=Config.FIGURES_DIR / "roc_liver_multicategory.pdf")plot_pr_grid(model_results, suptitle="Liver Pathology — Precision-Recall",             save_path=Config.FIGURES_DIR / "pr_liver_multicategory.pdf")plot_cm_grid(model_results, suptitle="Liver Pathology — Confusion Matrices (Youden's J)",             save_path=Config.FIGURES_DIR / "cm_liver_multicategory.pdf")plot_boxplot_grid(model_results, suptitle="Liver Pathology — Box Plots",                  save_path=Config.FIGURES_DIR / "boxplot_liver_multicategory.pdf")